<a href="https://colab.research.google.com/github/BrundaSreedhar/CPSC5310-Machine-Learning/blob/main/EX11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from tensorflow.keras.datasets import fashion_mnist

# Load the Fashion MNIST dataset
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

print(f"Original training data shape: {X_train_full.shape}")
print(f"Original test data shape: {X_test.shape}")

Original training data shape: (60000, 28, 28)
Original test data shape: (10000, 28, 28)


### Normalize the pixel values
Pixel values are typically between 0 and 255. Normalizing them to a range of 0 to 1 helps in faster and more stable training of neural networks.

In [4]:
# Normalize pixel values to be between 0 and 1
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

print(f"Normalized training data min: {X_train_full.min()}, max: {X_train_full.max()}")

Normalized training data min: 0.0, max: 1.0


### Split the training data into Training and Validation sets
It's good practice to have a separate validation set to monitor the model's performance during training and tune hyperparameters without touching the test set.

In [5]:
from sklearn.model_selection import train_test_split

# Split the full training data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_valid.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (48000, 28, 28)
Validation set shape: (12000, 28, 28)
Test set shape: (10000, 28, 28)


### Flatten the images
For a Multi-Layer Perceptron (MLP), input images need to be flattened from a 2D array (e.g., 28x28) into a 1D array (e.g., 784).

In [6]:
# Flatten the images for MLP input
# X_train is (num_samples, 28, 28) -> (num_samples, 28 * 28)
X_train_flattened = X_train.reshape(-1, 28 * 28)
X_valid_flattened = X_valid.reshape(-1, 28 * 28)
X_test_flattened = X_test.reshape(-1, 28 * 28)

print(f"Flattened training set shape: {X_train_flattened.shape}")
print(f"Flattened validation set shape: {X_valid_flattened.shape}")
print(f"Flattened test set shape: {X_test_flattened.shape}")

Flattened training set shape: (48000, 784)
Flattened validation set shape: (12000, 784)
Flattened test set shape: (10000, 784)


#PART 2

## Part 2 — Train a Baseline Deep MLP

Now that the data is prepared, let's build and train a Deep Multi-Layer Perceptron (MLP) classifier using Keras. The goal is to achieve at least 85% test accuracy by tuning hyperparameters.

In [7]:
import tensorflow as tf
from tensorflow import keras
import time

# Define the model architecture
# You can experiment with the number of hidden layers and neurons
model = keras.models.Sequential([
    keras.layers.Dense(300, activation="relu", input_shape=[784]), # First hidden layer
    keras.layers.Dense(100, activation="relu"), # Second hidden layer
    keras.layers.Dense(10, activation="softmax") # Output layer for 10 classes
])

# Display model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 266,610 (1.02 MB)

 Trainable params: 266,610 (1.02 MB)

 Non-trainable params: 0 (0.00 B)

### Compile the model
Choose an optimizer, loss function, and metrics. You can experiment with different optimizers and learning rates.

In [8]:
# Compile the model
# Experiment with different optimizers (e.g., 'adam', 'sgd') and learning rates
optimizer = keras.optimizers.SGD(learning_rate=0.01) # You can try Adam as well
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

### Train the model
Train the model using the training and validation sets. You can experiment with the number of epochs and batch size.

In [9]:
# Train the model
# Experiment with batch_size and epochs
start_time = time.time()
history = model.fit(X_train_flattened, y_train, epochs=30, # Number of epochs
                    validation_data=(X_valid_flattened, y_valid), # Validation data
                    batch_size=32) # Batch size
end_time = time.time()

training_time = end_time - start_time
print(f"Training time: {training_time:.2f} seconds")

Epoch 1/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7609 - loss: 0.7438 - val_accuracy: 0.8145 - val_loss: 0.5405
Epoch 2/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8277 - loss: 0.4970 - val_accuracy: 0.8350 - val_loss: 0.4706
Epoch 3/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8435 - loss: 0.4492 - val_accuracy: 0.8399 - val_loss: 0.4587
Epoch 4/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8524 - loss: 0.4217 - val_accuracy: 0.8553 - val_loss: 0.4139
Epoch 5/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8599 - loss: 0.4033 - val_accuracy: 0.8612 - val_loss: 0.4045
Epoch 6/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8652 - loss: 0.3849 - val_accuracy: 0.8638 - val_loss: 0.3889
Epoch 7/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8685 - loss: 0.3722 - val_accuracy: 0.8669 - val_loss: 0.3777
Epoch 8/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8729 - loss: 0.3588 -

### Evaluate the model on the test set
After training, evaluate the model's performance on unseen test data.

In [10]:
# Evaluate the model on the test set
# This gives you the final test accuracy
test_loss, test_accuracy = model.evaluate(X_test_flattened, y_test)

print(f"Final Test Accuracy: {test_accuracy:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8763 - loss: 0.3479
Final Test Accuracy: 0.8763


### Results

*   **Final test accuracy:** 0.88
*   **Training time:** 231s

## Part 5 — Reflection

### Write a short reflection (1–2 paragraphs):

*   **What hyperparameters had the largest impact on performance?**
    Based on the Keras Tuner results, the choice of optimizer (Adam over SGD), the learning rate (0.001), and surprisingly, a simpler network architecture (only one hidden layer with 384 neurons) seemed to have the largest impact. The initial baseline used two hidden layers and SGD, and the automated search found a slightly better configuration with fewer layers but a more advanced optimizer and a different learning rate. This suggests that the optimizer and learning rate are often critical, and sometimes simpler models can perform better if tuned correctly.

*   **What did you learn about training deep neural networks?**
    Training deep neural networks is not just about stacking many layers; it's crucially about finding the right balance of architectural complexity and optimization parameters. This exercise reinforced the importance of:
    1.  **Optimizer and Learning Rate**: Their influence on convergence and final performance is substantial.
    2.  **Regularization**: The best tuned model did not use dropout, suggesting that for this specific dataset and model size, it might not have been necessary or might have even hindered performance.
    3.  **Value of Validation Sets and Callbacks**: Early stopping and model checkpoints are indispensable for efficient and effective training, preventing overfitting, and saving the best model without manual intervention.

*   **Did automated tuning perform better than manual tuning?**
    Yes, automated tuning with Keras Tuner performed marginally better in terms of test accuracy (0.8788 vs. 0.8763). Beyond just the slight accuracy gain, the primary benefit was the automated exploration of a hyperparameter space that would be laborious and time-consuming to search manually. It efficiently identified a slightly superior configuration, demonstrating the power of automated tools in optimizing neural networks.

In [11]:
# Install Keras Tuner if not already installed
!pip install keras-tuner -q

## Part 3 — Automated Hyperparameter Tuning with Keras Tuner

Now, we will use Keras Tuner to automate the search for a better model by tuning various hyperparameters.

In [12]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras
import os
import datetime

# Define the model building function for Keras Tuner
def build_model(hp):
    model = keras.models.Sequential()
    model.add(keras.layers.InputLayer(input_shape=[784]))

    # Tune the number of hidden layers
    for i in range(hp.Int('num_hidden_layers', min_value=1, max_value=5, step=1)):
        # Tune the number of neurons in each dense layer
        model.add(keras.layers.Dense(hp.Int(f'n_neurons_layer_{i}', min_value=32, max_value=512, step=32),
                                     activation='relu'))
        # Optional: Add dropout layer
        if hp.Boolean(f'dropout_layer_{i}'):
            model.add(keras.layers.Dropout(hp.Float(f'dropout_rate_{i}', min_value=0.0, max_value=0.5, step=0.1)))

    model.add(keras.layers.Dense(10, activation='softmax')) # Output layer

    # Tune the optimizer and learning rate
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer_choice = hp.Choice('optimizer', values=['adam', 'sgd'])

    if optimizer_choice == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)

    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model


### Run Keras Tuner
We will use the RandomSearch, which is an efficient tuning algorithm.

In [13]:
#implement random search

### Setup Callbacks for Keras Tuner
We'll use Early Stopping, Model Checkpoints, and TensorBoard to monitor and optimize the training process during the hyperparameter search.

In [14]:
import os
import datetime
import tensorflow as tf
from tensorflow import keras

# Define callbacks

# Early Stopping: Stop training when validation loss stops improving
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

# Model Checkpoint: Save the best model during training
checkpoint_filepath = 'best_model.keras'
model_checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_best_only=True, # Only save a model if `val_loss` has improved.
    monitor='val_loss',
    mode='min',
    verbose=1
)

# TensorBoard: Log training metrics for visualization
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_cb = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

callbacks = [early_stopping_cb, model_checkpoint_cb, tensorboard_cb]


### Run Keras Tuner using RandomSearch
We will use the RandomSearch tuner for efficient hyperparameter exploration.

In [15]:
import keras_tuner as kt

# Instantiate the Keras Tuner RandomSearch
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,  # Number of different hyperparameter combinations to try
    executions_per_trial=1, # Number of models to train for each trial (set to 1 to speed up)
    directory='keras_tuner_dir',
    project_name='fashion_mnist_random_search',
    overwrite=True
)

# Run the hyperparameter search
print("Starting RandomSearch hyperparameter search...")
tuner.search(
    X_train_flattened, y_train,
    epochs=10, # Max epochs for each trial (will be stopped early by EarlyStopping if no improvement)
    validation_data=(X_valid_flattened, y_valid),
    callbacks=callbacks,
    batch_size=32 # Using a fixed batch size for tuning to simplify
)

print("RandomSearch hyperparameter search complete.")

# Get the optimal hyperparameters and best model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.get_best_models(num_models=1)[0]

print(f"Best hyperparameters found: {best_hps.values}")

Trial 10 Complete [00h 02m 33s]
val_accuracy: 0.7884166836738586

Best val_accuracy So Far: 0.8864166736602783
Total elapsed time: 00h 20m 45s
RandomSearch hyperparameter search complete.
Best hyperparameters found: {'num_hidden_layers': 1, 'n_neurons_layer_0': 384, 'dropout_layer_0': False, 'learning_rate': 0.001, 'optimizer': 'adam', 'n_neurons_layer_1': 448, 'dropout_layer_1': False, 'n_neurons_layer_2': 320, 'dropout_layer_2': True, 'n_neurons_layer_3': 224, 'dropout_layer_3': True, 'dropout_rate_1': 0.2, 'dropout_rate_2': 0.0, 'dropout_rate_3': 0.4, 'n_neurons_layer_4': 96, 'dropout_layer_4': True, 'dropout_rate_0': 0.2}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### Evaluate the Best Model from RandomSearch
Finally, evaluate the performance of the best model found by Keras Tuner's RandomSearch on the test set.

In [16]:
import time

start_time_best_model_eval = time.time()
test_loss_tuned, test_accuracy_tuned = best_model.evaluate(X_test_flattened, y_test)
end_time_best_model_eval = time.time()

print(f"Final Test Accuracy of Tuned Model: {test_accuracy_tuned:.4f}")
print(f"Time taken for best model evaluation: {end_time_best_model_eval - start_time_best_model_eval:.2f} seconds")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8788 - loss: 0.3399
Final Test Accuracy of Tuned Model: 0.8788
Time taken for best model evaluation: 1.32 seconds


## Part 4 — Compare Your Models

Let's compare the performance of the Baseline MLP and the Keras Tuner Model.

| Model             | Accuracy | Training Time | Notes |
|-------------------|----------|---------------|-------|
| Baseline MLP      | 0.8763   | 227.77s       | Initial manual tuning |
| Keras Tuner Model | 0.8788   | 20m 45s       | Automated tuning with RandomSearch |

### Answer:

*   **Did automated tuning outperform manual tuning?**
    Yes, automated tuning with Keras Tuner slightly outperformed the manual tuning in terms of final test accuracy. The Baseline MLP achieved a test accuracy of 0.8763, while the Keras Tuner Model achieved 0.8788.

*   **Which hyperparameters had the biggest impact?**
    The optimal hyperparameters identified by Keras Tuner included:
    *   `num_hidden_layers`: 1
    *   `n_neurons_layer_0`: 384
    *   `optimizer`: 'adam'
    *   `learning_rate`: 0.001
    *   `dropout_layer_0`: False
    It seems simplifying the architecture (fewer layers), combined with the Adam optimizer and a specific learning rate, had the most significant impact.